# RQ5: Sensitivity to Evaluation Metrics
**Research Question:** How does the relative ranking of candidate models change when different evaluation metrics are considered?

**Metrics:** MAE, RMSE, R²  
**Models:** Linear Regression, Random Forest, XGBoost, SVR

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import os

os.makedirs('figures', exist_ok=True)
os.makedirs('tables', exist_ok=True)

In [2]:
DATA_PATH = '/kaggle/input/datasets/karansridhar/karan-ue-ml-assignment-dataset/marketing_and_product_performance.csv'
df = pd.read_csv(DATA_PATH)
TARGET = 'Revenue_Generated'

id_cols = [c for c in df.columns if 'ID' in c or 'id' in c.lower()]
df = df.drop(columns=id_cols, errors='ignore').dropna(subset=[TARGET])
cat_cols = df.select_dtypes(include='object').columns
le = LabelEncoder()
for c in cat_cols:
    df[c] = le.fit_transform(df[c].fillna('Unknown').astype(str))

X = df.drop(columns=[TARGET])
y = df[TARGET]
imputer = SimpleImputer(strategy='median')
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

X_train, X_test, y_train, y_test = train_test_split(X_imp, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc  = scaler.transform(X_test)

In [3]:
models = {
    'Linear Regression': (LinearRegression(), True),
    'Random Forest':     (RandomForestRegressor(n_estimators=100, random_state=42), False),
    'XGBoost':           (XGBRegressor(n_estimators=100, random_state=42, verbosity=0), False),
    'SVR':               (SVR(kernel='rbf', C=10), True)
}

scores = {}
for name, (model, scaled) in models.items():
    Xtr = X_tr_sc if scaled else X_train.values
    Xte = X_te_sc  if scaled else X_test.values
    model.fit(Xtr, y_train)
    p = model.predict(Xte)
    scores[name] = {
        'MAE':  round(mean_absolute_error(y_test, p), 4),
        'RMSE': round(np.sqrt(mean_squared_error(y_test, p)), 4),
        'R²':   round(r2_score(y_test, p), 4)
    }

df_scores = pd.DataFrame(scores).T.reset_index().rename(columns={'index': 'Model'})

# Rank: lower MAE/RMSE = better (rank 1), higher R² = better (rank 1)
df_ranks = df_scores[['Model']].copy()
df_ranks['Rank by MAE']  = df_scores['MAE'].rank().astype(int)
df_ranks['Rank by RMSE'] = df_scores['RMSE'].rank().astype(int)
df_ranks['Rank by R²']   = df_scores['R²'].rank(ascending=False).astype(int)

df_ranks.to_csv('tables/RQ5_Table5_Metric_Sensitivity.csv', index=False)
print(df_ranks)

               Model  Rank by MAE  Rank by RMSE  Rank by R²
0  Linear Regression            2             2           2
1      Random Forest            3             3           3
2            XGBoost            4             4           4
3                SVR            1             1           1


In [4]:
# Figure 5 — Slope / Bump Chart
metrics = ['Rank by MAE', 'Rank by RMSE', 'Rank by R²']
colors = ['#1E88E5', '#43A047', '#E53935', '#FB8C00']

fig, ax = plt.subplots(figsize=(9, 6))
for i, (_, row) in enumerate(df_ranks.iterrows()):
    ranks = [row[m] for m in metrics]
    ax.plot(range(len(metrics)), ranks, marker='o', linewidth=2.5,
            color=colors[i], label=row['Model'], markersize=9)
    ax.text(len(metrics)-1 + 0.05, ranks[-1], row['Model'], va='center',
            fontsize=10, color=colors[i])

ax.set_xticks(range(len(metrics)))
ax.set_xticklabels([m.replace('Rank by ', '') for m in metrics], fontsize=12)
ax.set_ylabel('Rank (1 = best)', fontsize=12)
ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.invert_yaxis()
ax.set_title('Figure 5: Model Rankings Across Evaluation Metrics\n(Marketing and Product Performance Dataset)', fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('figures/RQ5_Figure5_Metric_Sensitivity.pdf', bbox_inches='tight')
plt.show()
print('Figure saved.')

Figure saved.
